In [8]:
from sklearn.neighbors import kneighbors_graph
import pandas as pd
import numpy as np
import plotly.graph_objects as go

import os
import pickle
from contextlib import nullcontext
import torch
import tiktoken
from model import GPTConfig, GPT

In [2]:
num_layers = 7
emb_dim = 384

In [4]:
from app import *

Loading model...
number of parameters: 29.94M
Model loaded. Loading encode and decode functions...
Encode and decode loaded. Loading precomputed data...
Loaded precomputed data. Computing 3d reductions...


/Users/isabella_zhu/urop/website/env/lib/python3.12/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


Computed 3D reductions.


In [6]:
import requests

url = "https://drive.google.com/file/d/1Wm5kDVliTpxsQ4bLXLdCH0S_0VJQlHF4/view?usp=sharing"
response = requests.get(url, stream=True)
response.raise_for_status()  # Ensure we got the file

# Save the downloaded content to check its validity
with open("downloaded_ckpt.pt", "wb") as f:
    f.write(response.content)

print("File downloaded. Check 'downloaded_ckpt.pt' to ensure it's a valid PyTorch model.")


File downloaded. Check 'downloaded_ckpt.pt' to ensure it's a valid PyTorch model.


In [7]:
import gdown
import torch
import io

file_id = "1Wm5kDVliTpxsQ4bLXLdCH0S_0VJQlHF4"
url = f"https://drive.google.com/uc?id={file_id}"

# Download the file into memory
response = requests.get(url)
response.raise_for_status()

checkpoint = torch.load(io.BytesIO(response.content), map_location="cpu")

UnpicklingError: invalid load key, '<'.

In [9]:
url = "https://www.dropbox.com/scl/fi/ocaovecmf7che47p1pn0o/ckpt.pt?rlkey=lpfz7b1e5k26ypuw6gltocxub&st=mt9nen3h&dl=1"

In [10]:
import requests
import torch
import io

# Download the file
response = requests.get(url, stream=True)
response.raise_for_status()  # Ensure the request was successful

# Load model from memory
checkpoint = torch.load(io.BytesIO(response.content), map_location="cpu")

print("Model loaded successfully!")


Model loaded successfully!


In [11]:
import numpy as np
import os
import io
import requests

In [14]:
url = "https://www.dropbox.com/scl/fi/ocaovecmf7che47p1pn0o/ckpt.pt?rlkey=lpfz7b1e5k26ypuw6gltocxub&st=mt9nen3h&dl=1"

In [17]:
def load_model():
    response = requests.get(url, stream=True)
    response.raise_for_status()  # Ensure the request was successful
    checkpoint = torch.load(io.BytesIO(response.content), map_location="cpu")

    # Load the model directly from memory (RAM)
    # ckpt_path = os.path.join('out-tinystories', 'ckpt.pt')
    # checkpoint = torch.load(ckpt_path, map_location=torch.device("cpu"))  # Ensure model loads on CPU
    
    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)
    model.to("cpu")  # Move model to CPU
    state_dict = checkpoint['model']

    # Remove unwanted prefixes in state dict
    unwanted_prefix = '_orig_mod.'
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model.load_state_dict(state_dict)
    model.eval()  # Set model to evaluation mode
    model.to("cpu")  # Move model to CPU explicitly

    return model

def get_encode_decode():
    enc = tiktoken.get_encoding("gpt2")
    encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
    decode = lambda l: enc.decode(l)
    return encode, decode

In [18]:
print("Loading model...")
model = load_model()
print("Model loaded. Loading encode and decode functions...")
encode, decode = get_encode_decode()
print("Encode and decode loaded.")

Loading model...
number of parameters: 29.94M
Model loaded. Loading encode and decode functions...
Encode and decode loaded.


In [28]:
def get_embeddings(text):
    start_ids = encode(text)
    x = torch.tensor(start_ids, dtype=torch.long, device="cpu")[None, ...]

    embs = model.get_embeddings(x)
    return [emb.detach().cpu().tolist() for emb in embs], [decode([token]) for token in start_ids]


In [29]:
get_embeddings("hello hello hello here is some random text i don't really care about")

([[[0.009349443018436432,
    -0.011439952999353409,
    -0.011984210461378098,
    -0.06535901874303818,
    -0.01077546551823616,
    -0.13836103677749634,
    -0.0068316273391246796,
    -0.03692307695746422,
    -0.12229454517364502,
    0.03642432391643524,
    0.05461468920111656,
    0.006214838474988937,
    -0.05237068608403206,
    -0.09039053320884705,
    0.05018473044037819,
    0.12222205102443695,
    -0.043184150010347366,
    -0.06323472410440445,
    0.1255262792110443,
    -0.05827977880835533,
    0.033473558723926544,
    -0.031129468232393265,
    -0.005733799189329147,
    -0.09883145242929459,
    -0.03587326407432556,
    -0.09807801246643066,
    0.07197067141532898,
    0.07116631418466568,
    -0.0883767381310463,
    0.03931374102830887,
    0.018431928008794785,
    0.04904768615961075,
    0.021144811064004898,
    -0.1160060465335846,
    -0.005925426725298166,
    0.006430604495108128,
    0.011600397527217865,
    0.02537619322538376,
    -0.0282907783